<a href="https://colab.research.google.com/github/pushpakgaidhane294/NNDL_lab/blob/main/Practical10_NNDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# ============================================================
# DEEP LEARNING BASED RECOMMENDATION SYSTEM
# Dataset: Online Retail Sales Data
# ============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ============================================================
# 1. LOAD DATASET
# ============================================================

FILE_PATH = "Online Retail.xlsx"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {FILE_PATH}\n"
        "Keep 'Online Retail.xlsx' in the same folder as this Python file."
    )

df = pd.read_excel(FILE_PATH)

print("=" * 70)
print("DEEP LEARNING BASED RECOMMENDATION SYSTEM")
print("=" * 70)

print("\nDataset loaded successfully.")
print("Original dataset shape:", df.shape)


# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "CustomerID",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# 3. DATA CLEANING
# ============================================================

df = df.dropna(
    subset=[
        "CustomerID",
        "StockCode",
        "Quantity",
        "UnitPrice"
    ]
)

df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

df["CustomerID"] = df["CustomerID"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

# Clean product descriptions
df["Description"] = (
    df["Description"]
    .astype(str)
    .str.strip()
)

print("Cleaned dataset shape:", df.shape)


# ============================================================
# 4. CREATE PRODUCT INFORMATION
# ============================================================

product_info = (
    df[["StockCode", "Description"]]
    .drop_duplicates(subset=["StockCode"])
    .copy()
)

product_info["Description"] = (
    product_info["Description"]
    .replace("nan", "Unknown Product")
)


# ============================================================
# 5. CREATE CUSTOMER-PRODUCT PURCHASE DATA
# ============================================================

data = (
    df.groupby(
        ["CustomerID", "StockCode"],
        as_index=False
    )["Quantity"]
    .sum()
)

print("Customer-Product interactions:", len(data))


# ============================================================
# 6. ENCODE CUSTOMERS AND PRODUCTS
# ============================================================

customer_encoder = LabelEncoder()
product_encoder = LabelEncoder()

data["CustomerIndex"] = customer_encoder.fit_transform(
    data["CustomerID"]
)

data["ProductIndex"] = product_encoder.fit_transform(
    data["StockCode"]
)


# ============================================================
# 7. CREATE TARGET / PURCHASE STRENGTH
# ============================================================

# Log transformation reduces the effect of very large quantities
data["PurchaseStrength"] = np.log1p(
    data["Quantity"]
)

max_strength = data["PurchaseStrength"].max()

data["Target"] = (
    data["PurchaseStrength"] / max_strength
)


# ============================================================
# 8. TRAIN-TEST SPLIT
# ============================================================

X = data[
    ["CustomerIndex", "ProductIndex"]
]

y = data["Target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 9. BUILD DEEP LEARNING MODEL
# ============================================================

num_customers = data["CustomerIndex"].nunique()
num_products = data["ProductIndex"].nunique()

embedding_size = 32


# Customer input
customer_input = tf.keras.Input(
    shape=(1,),
    name="Customer_Input"
)

# Product input
product_input = tf.keras.Input(
    shape=(1,),
    name="Product_Input"
)


# Customer embedding
customer_embedding = tf.keras.layers.Embedding(
    input_dim=num_customers,
    output_dim=embedding_size,
    name="Customer_Embedding"
)(customer_input)

# Product embedding
product_embedding = tf.keras.layers.Embedding(
    input_dim=num_products,
    output_dim=embedding_size,
    name="Product_Embedding"
)(product_input)


# Flatten embeddings
customer_vector = tf.keras.layers.Flatten()(
    customer_embedding
)

product_vector = tf.keras.layers.Flatten()(
    product_embedding
)


# Combine customer and product information
combined = tf.keras.layers.Concatenate()([
    customer_vector,
    product_vector
])


# Neural network layers
x = tf.keras.layers.Dense(
    128,
    activation="relu"
)(combined)

x = tf.keras.layers.Dropout(0.30)(x)

x = tf.keras.layers.Dense(
    64,
    activation="relu"
)(x)

x = tf.keras.layers.Dropout(0.20)(x)

x = tf.keras.layers.Dense(
    32,
    activation="relu"
)(x)


# Output purchase strength
output = tf.keras.layers.Dense(
    1,
    activation="linear",
    name="Purchase_Strength"
)(x)


# Create model
model = tf.keras.Model(
    inputs=[
        customer_input,
        product_input
    ],
    outputs=output
)


# ============================================================
# 10. COMPILE MODEL
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


print("\n" + "=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)

model.summary()


# ============================================================
# 11. TRAIN MODEL
# ============================================================

print("\n" + "=" * 70)
print("MODEL TRAINING")
print("=" * 70)

history = model.fit(
    [
        X_train["CustomerIndex"].values,
        X_train["ProductIndex"].values
    ],
    y_train.values,
    epochs=15,
    batch_size=256,
    validation_split=0.20,
    verbose=1
)


# ============================================================
# 12. MODEL EVALUATION
# ============================================================

predictions = model.predict(
    [
        X_test["CustomerIndex"].values,
        X_test["ProductIndex"].values
    ],
    verbose=0
).flatten()

mse = mean_squared_error(
    y_test,
    predictions
)

rmse = np.sqrt(mse)

mae = mean_absolute_error(
    y_test,
    predictions
)


print("\n" + "=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

print(f"Mean Squared Error (MSE)  : {mse:.4f}")
print(f"Root Mean Squared Error   : {rmse:.4f}")
print(f"Mean Absolute Error (MAE) : {mae:.4f}")


# ============================================================
# 13. SELECT CUSTOMER FOR RECOMMENDATION
# ============================================================

customer_counts = (
    data.groupby("CustomerID")
    .size()
    .sort_values(ascending=False)
)

selected_customer = customer_counts.index[0]

customer_index = customer_encoder.transform(
    [selected_customer]
)[0]


# ============================================================
# 14. FIND ALREADY PURCHASED PRODUCTS
# ============================================================

purchased_products = set(
    data[
        data["CustomerID"] == selected_customer
    ]["StockCode"]
)


# ============================================================
# 15. FIND PRODUCTS NOT YET PURCHASED
# ============================================================

all_products = set(
    data["StockCode"]
)

candidate_products = list(
    all_products - purchased_products
)


# ============================================================
# 16. PREDICT RECOMMENDATION SCORES
# ============================================================

candidate_product_indices = product_encoder.transform(
    candidate_products
)

customer_indices = np.full(
    len(candidate_products),
    customer_index
)


predicted_scores = model.predict(
    [
        customer_indices,
        candidate_product_indices
    ],
    verbose=0
).flatten()


# Keep scores in valid range
predicted_scores = np.clip(
    predicted_scores,
    0,
    1
)


# ============================================================
# 17. CREATE RECOMMENDATION DATAFRAME
# ============================================================

recommendations = pd.DataFrame({
    "StockCode": candidate_products,
    "Recommendation Score": predicted_scores
})


# Sort highest score first
recommendations = recommendations.sort_values(
    by="Recommendation Score",
    ascending=False
)


# Select Top 10
top_recommendations = recommendations.head(10).copy()


# ============================================================
# 18. ADD PRODUCT NAMES
# ============================================================

top_recommendations = top_recommendations.merge(
    product_info,
    on="StockCode",
    how="left"
)


top_recommendations = top_recommendations.rename(
    columns={
        "Description": "Product Name"
    }
)


# Arrange columns
top_recommendations = top_recommendations[
    [
        "Product Name",
        "StockCode",
        "Recommendation Score"
    ]
]


# ============================================================
# 19. DISPLAY CUSTOMER INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("CUSTOMER SELECTED FOR RECOMMENDATION")
print("=" * 70)

print(f"Customer ID              : {selected_customer}")
print(f"Previously Purchased     : {len(purchased_products)} products")


# ============================================================
# 20. DISPLAY FINAL RECOMMENDATIONS
# ============================================================

print("\n" + "=" * 70)
print("TOP-10 PRODUCT RECOMMENDATIONS")
print("=" * 70)

print(
    top_recommendations.to_string(
        index=False,
        formatters={
            "Recommendation Score": "{:.4f}".format
        }
    )
)


# ============================================================
# 21. FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)

print(
    "A deep learning-based recommendation system was successfully "
    "developed using customer-product sales data."
)

print(
    "The model uses customer and product embeddings to learn "
    "purchasing patterns and recommend products."
)

print(
    f"Top-{len(top_recommendations)} personalized product "
    "recommendations were generated successfully."
)

print("=" * 70)

DEEP LEARNING BASED RECOMMENDATION SYSTEM

Dataset loaded successfully.
Original dataset shape: (541909, 8)
Cleaned dataset shape: (397884, 8)
Customer-Product interactions: 266792

MODEL ARCHITECTURE


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Customer_Input      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Product_Input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Customer_Embedding  │ (None, 1, 32)     │    138,816 │ Customer_Input[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Product_Embedding   │ (None, 1, 32)     │    117,280 │ Product_Input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_10          │ (None, 32)        │          0 │ Customer_Embeddi… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_11          │ (None, 32)        │          0 │ Product_Embeddin… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 64)        │          0 │ flatten_10[0][0], │
│ (Concatenate)       │                   │            │ flatten_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 128)       │      8,320 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 128)       │          0 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 64)        │      8,256 │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 64)        │          0 │ dense_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 32)        │      2,080 │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Purchase_Strength   │ (None, 1)         │         33 │ dense_18[0][0]    │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 274,785 (1.05 MB)

 Trainable params: 274,785 (1.05 MB)

 Non-trainable params: 0 (0.00 B)


MODEL TRAINING
Epoch 1/15
667/667 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.0053 - mae: 0.0545 - val_loss: 0.0037 - val_mae: 0.0449
Epoch 2/15
667/667 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.0036 - mae: 0.0444 - val_loss: 0.0035 - val_mae: 0.0437
Epoch 3/15
667/667 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.0033 - mae: 0.0423 - val_loss: 0.0035 - val_mae: 0.0428
Epoch 4/15
667/667 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 0.0031 - mae: 0.0409 - val_loss: 0.0034 - val_mae: 0.0430
Epoch 5/15
667/667 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - loss: 0.0029 - mae: 0.0397 - val_loss: 0.0034 - val_mae: 0.0423
Epoch 6/15
667/667 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0027 - mae: 0.0385 - val_loss: 0.0034 - val_mae: 0.0422
Epoch 7/15
667/667 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - loss: 0.0026 - mae: 0.0373 - val_loss: 0.0034 - val_mae: 0.0424
Epoch 8/15
667/667 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - loss: 0.0024 - mae: 0.0364 - val_loss: 0.0033 - val_mae: 0.0419
Epoch 9/15
667/667 ━━━━━━━━━━━━━━━━━━━